# ONDO 가상 피팅 이미지 일괄 생성 (CatVTON · Kaggle 무료 GPU)

**한 장에 한 벌씩만** 입힙니다. 상의·하의·아우터·신발·가방을 조합하지 않습니다.

모든 결과는 **기본 모델 6장(`{체형}_{톤}_lower_base.png`)** 위에 생성합니다. 모델이 직접 입고·신고·드는 이미지로 만듭니다.

| 카테고리 | 마스크 |
|---|---|
| 상의 | upper (크롭탑 자리) |
| 아우터 | outer |
| 하의(스커트·팬츠·쇼츠) | lower → 쇼츠까지 덮어서 치마는 맨다리 위에 |
| 신발 | 발 영역 (맨발 위에 신긴 모습으로 생성) |
| 가방 | 허리 옆·손 영역 (모델이 든 모습으로 생성) |

## 실행 방법
1. 오른쪽 **Settings → Accelerator: GPU T4 x2(또는 P100)**, **Internet: On** (전화번호 인증 필요)
2. **Add Input**으로 `ondo-tryon-kit` 데이터셋(이 zip을 올린 것)을 추가
3. 아래 `설정` 셀에서 만들 범위를 고르고 **Run All**
4. 끝나면 오른쪽 **Output**에서 `ondo_fitting_results.zip`을 내려받아 Claude에게 전달

세션이 끊겨도 이어서 만들 수 있습니다. 이전 실행의 Output을 Input으로 추가하면 이미 만든 조합은 건너뜁니다.

In [ ]:
# ===== 설정 =====
PRIORITY_MAX = 1            # 1 = 상품마다 대표 색상 1개만 (1590장)
CATEGORIES = ["top", "bottom", "outer"]   # 신발·가방은 테스트 품질 미달 → 이번 실행 제외
BODIES = ["straight", "wave", "natural"]
TONES = ["warm", "cool"]
STEPS = 30
GUIDANCE = 2.5
SEED = 42
LIMIT = None
WIDTH, HEIGHT = 768, 1024
TEST_IDS = None
TIME_BUDGET_MIN = 660   # 이 시간이 지나면 새 작업을 멈추고 지금까지 만든 것만 저장
STALL_MIN = 15                 # 한 단계가 이 시간 넘게 멈추면 위치를 출력하고 강제 종료


In [ ]:
import subprocess, sys, os, importlib.metadata as md
if not os.path.exists("/tmp/CatVTON"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Zheng-Chong/CatVTON.git", "/tmp/CatVTON"], check=True)
# 기본 설치된 numpy·numba는 버전을 그대로 고정 (지난번엔 numpy가 2.5로 올라가며 꼬임)
pins = []
for p in ["numpy", "numba", "torch", "torchvision"]:
    try: pins.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: pass
open("/tmp/constraints.txt", "w").write("\n".join(pins) + "\n"); print("고정:", pins)
# CatVTON이 동작하는 버전 조합으로 고정 (transformers 5.x에서 모델 로딩 중 멈춤)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/constraints.txt",
                "diffusers==0.31.0", "transformers==4.46.3", "huggingface_hub==0.26.5", "accelerate==1.1.1",
                "fvcore", "cloudpickle", "omegaconf", "pycocotools", "av", "rembg==2.0.59", "onnxruntime"], check=True)
for p in ["numpy", "torch", "diffusers", "transformers", "huggingface_hub", "rembg"]:
    print(p, md.version(p))
sys.path.insert(0, "/tmp/CatVTON")
print("ready")


In [ ]:
WORKER_SRC = "import os, sys, csv, glob, time, json, threading, faulthandler\nos.environ.update(HF_HUB_DISABLE_PROGRESS_BARS=\"1\", TRANSFORMERS_VERBOSITY=\"error\", DIFFUSERS_VERBOSITY=\"error\", TQDM_DISABLE=\"1\")\nSHARD, NSHARD = int(sys.argv[1]), int(sys.argv[2])\ncfg = json.load(open(\"/kaggle/working/run_config.json\"))\nglobals().update(cfg)\nT_START = time.time()\ndef log(*a): print(f\"[gpu{SHARD} {time.time()-T_START:6.0f}s]\", *a, flush=True)\n# ---- 멈춤 감시: 마지막 진행 후 STALL_MIN 분이 지나면 현재 코드 위치를 출력하고 종료 ----\n_beat = [time.time(), \"시작\"]\ndef beat(stage): _beat[0] = time.time(); _beat[1] = stage\ndef _watch():\n    while True:\n        time.sleep(30)\n        if time.time() - _beat[0] > STALL_MIN * 60:\n            log(f\"!!! '{_beat[1]}' 단계에서 {STALL_MIN}분 넘게 멈춤 → 위치 출력 후 종료\")\n            faulthandler.dump_traceback(all_threads=True); sys.stdout.flush(); os._exit(3)\nthreading.Thread(target=_watch, daemon=True).start()\nsys.path.insert(0, \"/tmp/CatVTON\")\nkit = os.path.dirname(glob.glob(\"/kaggle/input/**/jobs.csv\", recursive=True)[0])\nOUT = \"/kaggle/working/results\"; os.makedirs(OUT, exist_ok=True)\ndone = set()\nfor p in glob.glob(\"/kaggle/input/**/results/*/*.png\", recursive=True) + glob.glob(OUT + \"/*/*.png\"):\n    done.add(os.path.basename(os.path.dirname(p)) + \"_\" + os.path.splitext(os.path.basename(p))[0])\njobs = [r for r in csv.DictReader(open(f\"{kit}/jobs.csv\", encoding=\"utf-8\"))\n        if int(r[\"priority\"]) <= PRIORITY_MAX and r[\"category\"] in CATEGORIES and r[\"body\"] in BODIES and r[\"tone\"] in TONES]\nif TEST_IDS: jobs = [r for r in jobs if r[\"job_id\"] in TEST_IDS]\ntodo = [r for r in jobs if r[\"job_id\"] not in done]\nif LIMIT: todo = todo[:LIMIT]\ntodo = todo[SHARD::NSHARD]\nlog(f\"대상 {len(jobs)}장 / 이미 완료 {len(jobs)-len([r for r in jobs if r['job_id'] not in done])}장 / 이 GPU가 생성 {len(todo)}장\")\nif not todo: sys.exit(0)\nimport torch, numpy as np\nfrom PIL import Image, ImageFilter\nfrom huggingface_hub import snapshot_download\nfrom diffusers.image_processor import VaeImageProcessor\nfrom model.pipeline import CatVTONPipeline\nfrom model.cloth_masker import AutoMasker\nfrom utils import resize_and_padding\n\nbeat(\"CatVTON 가중치 다운로드\"); log(\"가중치 다운로드\")\nrepo = snapshot_download(\"zhengchong/CatVTON\")\nbeat(\"파이프라인 로딩\"); log(\"파이프라인 로딩\")\npipe = CatVTONPipeline(base_ckpt=\"booksforcharlie/stable-diffusion-inpainting\", attn_ckpt=repo, attn_ckpt_version=\"mix\",\n                       weight_dtype=torch.float16, use_tf32=False, device=\"cuda\", skip_safety_check=True)\nbeat(\"마스커 로딩\"); log(\"마스커(DensePose/SCHP) 로딩\")\nmasker = AutoMasker(densepose_ckpt=os.path.join(repo, \"DensePose\"), schp_ckpt=os.path.join(repo, \"SCHP\"), device=\"cuda\")\nmask_proc = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)\nbeat(\"배경제거 모델 로딩\"); log(\"배경제거 모델 로딩\")\nfrom rembg import remove, new_session\ncutter = new_session(\"u2net_human_seg\")\nlog(\"모델 로딩 완료\")\nBG = (255, 255, 255)\ndef on_white(im):\n    im = im.convert(\"RGBA\"); bg = Image.new(\"RGBA\", im.size, BG + (255,)); bg.alpha_composite(im); return bg.convert(\"RGB\")\n\n_avatar_cache = {}\ndef avatar(name):\n    if name not in _avatar_cache:\n        rgba = Image.open(f\"{kit}/avatars/{name}\").convert(\"RGBA\")\n        _avatar_cache[name] = (rgba, on_white(rgba).resize((WIDTH, HEIGHT), Image.LANCZOS))\n    return _avatar_cache[name]\n\ndef body_alpha(rgba):\n    return (np.array(rgba.resize((WIDTH, HEIGHT)).split()[-1]) > 40)\n\ndef rect_mask(box):\n    m = np.zeros((HEIGHT, WIDTH), np.uint8); x0, y0, x1, y1 = [int(v) for v in box]\n    m[max(0,y0):min(HEIGHT,y1), max(0,x0):min(WIDTH,x1)] = 255; return Image.fromarray(m)\n\ndef accessory_mask(rgba, kind):\n    a = body_alpha(rgba); ys, xs = np.where(a); top, bot = ys.min(), ys.max(); H = bot - top\n    if kind == \"shoes\":                       # 발 영역 (발목 아래)\n        band = a[bot - int(H*.07):bot + 1]; cols = np.where(band.any(0))[0]\n        return rect_mask((cols.min() - 18, bot - int(H*.075), cols.max() + 18, min(HEIGHT, bot + 14)))\n    yh = top + int(H*.50); row = np.where(a[yh])[0]   # 가방: 허리 옆, 손 주변\n    return rect_mask((row.min() - int(H*.13), yh - int(H*.04), row.min() + int(H*.05), yh + int(H*.16)))\n\nSLEEVELESS = [\"나시\", \"민소매\", \"탱크\", \"밴도\", \"홀터\", \"원숄더\", \"코르셋\", \"베스트\", \"캐미\", \"슬리브리스\"]\nSHORT_SLEEVE = [\"반팔\", \"캠프 셔츠\", \"플러터소매\"]\ndef sleeve_of(job):\n    n = job[\"name\"]\n    if any(k in n for k in SLEEVELESS): return \"none\"\n    if any(k in n for k in SHORT_SLEEVE): return \"short\"\n    return \"long\"\n_pre_cache = {}; _mask_cache = {}\nDENSE_BIG_ARMS = [15, 16, 17, 18]; DENSE_FOREARMS = [19, 20, 21, 22]\ndef make_mask(job, rgba, person):\n    gt = job[\"garment_type\"]\n    if gt not in (\"upper\", \"outer\", \"lower\"):\n        return accessory_mask(rgba, gt)\n    sleeve = sleeve_of(job) if gt in (\"upper\", \"outer\") else \"-\"\n    key = (job[\"base_avatar\"], gt, sleeve, job[\"is_skirt\"] if gt == \"lower\" else \"-\")\n    if key in _mask_cache: return _mask_cache[key]\n    if job[\"base_avatar\"] not in _pre_cache:\n        _pre_cache[job[\"base_avatar\"]] = masker.preprocess_image(person)\n    pre = _pre_cache[job[\"base_avatar\"]]\n    m = masker.cloth_agnostic_mask(pre[\"densepose\"], pre[\"schp_lip\"], pre[\"schp_atr\"], part=gt)\n    arr = np.array(m.convert(\"L\").resize((WIDTH, HEIGHT), Image.NEAREST)) > 127\n    a = body_alpha(rgba); ys, _ = np.where(a); top, bot = ys.min(), ys.max(); H = bot - top\n    if gt == \"lower\":\n        # 맨다리 기본 모델의 쇼츠가 남지 않도록 쇼츠+다리를 확실히 덮는다 (치마는 맨다리 위에)\n        hip = top + int(H*.43); knee_floor = top + int(H*(.62 if job[\"is_skirt\"] == \"1\" else .9))\n        arr[hip:knee_floor] |= a[hip:knee_floor]\n    elif sleeve != \"long\":\n        # 민소매·반팔: 팔은 그대로 둔다 (팔 자리에 엉뚱한 소매·가디건이 생기는 문제 방지)\n        dp = np.array(pre[\"densepose\"].resize((WIDTH, HEIGHT), Image.NEAREST))\n        arms = np.isin(dp, DENSE_FOREARMS)\n        if sleeve == \"none\": arms |= np.isin(dp, DENSE_BIG_ARMS)\n        lip = np.array(pre[\"schp_lip\"].resize((WIDTH, HEIGHT), Image.NEAREST))\n        if sleeve == \"none\": arms |= np.isin(lip, [14, 15])\n        else: arms |= np.isin(lip, [14, 15]) & (np.arange(HEIGHT)[:, None] > top + int(H*.27))\n        arr &= ~arms\n    m = Image.fromarray((arr * 255).astype(np.uint8)); _mask_cache[key] = m\n    return m\nimport time, zipfile\nmanifest_path = f\"{OUT}/manifest_{SHARD}.csv\"\nfields = [\"job_id\",\"body\",\"tone\",\"variant_id\",\"product_id\",\"category\",\"subtype\",\"name\",\"color\",\"file\"]\nnew_file = not os.path.exists(manifest_path)\nmf = open(manifest_path, \"a\", newline=\"\", encoding=\"utf-8\"); mw = csv.DictWriter(mf, fieldnames=fields)\nif new_file: mw.writeheader()\ngen = torch.Generator(device=\"cuda\").manual_seed(SEED)\nt0 = time.time(); deadline = T_START + TIME_BUDGET_MIN * 60\nfor i, job in enumerate(todo, 1):\n    if time.time() > deadline:\n        log(f\"시간 예산 {TIME_BUDGET_MIN}분 도달 → 여기서 멈추고 저장 ({i-1}/{len(todo)})\"); break\n    beat(f\"{job['job_id']} 생성\"); ts = time.time()\n    try:\n        rgba, person = avatar(job[\"base_avatar\"])\n        cloth = resize_and_padding(on_white(Image.open(f\"{kit}/{job['image_path']}\")), (WIDTH, HEIGHT))\n        mask = mask_proc.blur(make_mask(job, rgba, person), blur_factor=9)\n        result = pipe(image=person, condition_image=cloth, mask=mask, num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=gen)[0]\n        result = result.resize(rgba.size, Image.LANCZOS)\n        cut = remove(result, session=cutter)              # 배경 제거 → 사이트 배경 위에 올리기\n        folder = f\"{OUT}/{job['body']}_{job['tone']}\"; os.makedirs(folder, exist_ok=True)\n        path = f\"{folder}/{job['variant_id']}.png\"; cut.save(path, optimize=True)\n        mw.writerow({k: job[k] for k in fields if k != \"file\"} | {\"file\": os.path.relpath(path, OUT)}); mf.flush()\n        if i <= 3: log(f\"{job['job_id']} 완료 {time.time()-ts:.1f}s\")\n    except Exception as e:\n        import traceback; log(\"실패\", job[\"job_id\"], repr(e)); traceback.print_exc(); sys.stdout.flush()\n    if i % 10 == 0 or i == len(todo):\n        el = time.time() - t0; log(f\"{i}/{len(todo)}  {el/i:.1f}s/장  남은 예상 {el/i*(len(todo)-i)/60:.0f}분\")\nmf.close(); log(\"끝\")\n"


In [ ]:
# ===== GPU 여러 장에 나눠서 동시에 생성 (시간 예산 + 멈춤 감시) =====
import json, subprocess, torch, time
json.dump(dict(PRIORITY_MAX=PRIORITY_MAX, CATEGORIES=CATEGORIES, BODIES=BODIES, TONES=TONES, STEPS=STEPS, GUIDANCE=GUIDANCE, SEED=SEED, LIMIT=LIMIT, TEST_IDS=TEST_IDS, WIDTH=WIDTH, HEIGHT=HEIGHT, TIME_BUDGET_MIN=TIME_BUDGET_MIN, STALL_MIN=STALL_MIN), open("/kaggle/working/run_config.json", "w"))
open("/kaggle/working/worker.py", "w", encoding="utf-8").write(WORKER_SRC)
n = max(1, torch.cuda.device_count()); print("GPU", n, "장", flush=True)
procs = [subprocess.Popen([sys.executable, "-u", "/kaggle/working/worker.py", str(i), str(n)], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i)}) for i in range(n)]
hard = time.time() + (TIME_BUDGET_MIN + 20) * 60   # 예산 + 20분이 지나도 안 끝나면 강제 종료
while any(p.poll() is None for p in procs) and time.time() < hard:
    time.sleep(20)
for p in procs:
    if p.poll() is None: p.kill(); print("강제 종료한 작업 있음", flush=True)
print("종료 코드", [p.wait() for p in procs], flush=True)


In [ ]:
import glob, csv, os
OUT = "/kaggle/working/results"
rows = []
for p in sorted(glob.glob(f"{OUT}/manifest_*.csv")):
    rows += list(csv.DictReader(open(p, encoding="utf-8")))
fields = ["job_id","body","tone","variant_id","product_id","category","subtype","name","color","file"]
with open(f"{OUT}/manifest.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)
manifest_path = f"{OUT}/manifest.csv"; print("생성 완료", len(rows), "장")
from PIL import Image


In [ ]:
# 검수용 미리보기: 카테고리별로 최대 12장
import random
rows = list(csv.DictReader(open(manifest_path, encoding="utf-8")))
for cat in CATEGORIES:
    pick = [r for r in rows if r["category"] == cat][:12]
    if not pick: continue
    sheet = Image.new("RGB", (6*272, ((len(pick)+5)//6)*362), (236, 228, 220))
    for k, r in enumerate(pick):
        im = Image.open(f"{OUT}/{r['file']}").convert("RGBA"); im.thumbnail((272, 362))
        sheet.paste(im, ((k%6)*272, (k//6)*362), im)
    sheet.save(f"{OUT}/preview_{cat}.jpg", quality=85); display(sheet.resize((sheet.width//2, sheet.height//2)))

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/ondo_fitting_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), "/kaggle/working"))
print("완료: Output 탭에서 ondo_fitting_results.zip 을 내려받으세요")